# Spaceship Titanic — Kaggle Competition
**Goal:** Predict which passengers were transported to an alternate dimension.

**Metric:** Classification Accuracy

**Pipeline:**
1. EDA
2. Data Cleaning & Feature Engineering
3. Encoding
4. Modeling (Logistic Regression → Random Forest → LightGBM)
5. Submission

## 1. Imports & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

## 2. EDA

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
# Missing values
print(train.isna().sum())

In [ ]:
# Target distribution — check if classes are balanced
print(train['Transported'].value_counts())

In [ ]:
# Transportation rate by HomePlanet
print(train.groupby('HomePlanet')['Transported'].mean())

In [ ]:
# CryoSleep vs spending — CryoSleep passengers spend nothing
services = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
train['totalSpend_eda'] = train[services].sum(axis=1)
print(train.groupby('CryoSleep')['totalSpend_eda'].mean())
train.drop('totalSpend_eda', axis=1, inplace=True)

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Split Cabin into Deck, RoomNum, Side
train[['Deck', 'RoomNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
test[['Deck', 'RoomNum', 'Side']] = test['Cabin'].str.split('/', expand=True)

# Split PassengerId into GroupId and PersonId
train[['groupId', 'personId']] = train['PassengerId'].str.split('_', expand=True)
test[['groupId', 'personId']] = test['PassengerId'].str.split('_', expand=True)

# Group size — computed from train only to avoid leakage
group_counts = train['groupId'].value_counts()
train['groupSize'] = train['groupId'].map(group_counts)
test['groupSize'] = test['groupId'].map(group_counts)

# isAlone feature — solo travelers may behave differently
train['isAlone'] = (train['groupSize'] == 1).astype(int)
test['isAlone'] = (test['groupSize'] == 1).astype(int)

# Save PassengerId before dropping
test_passenger_ids = test['PassengerId'].copy()

# Drop columns we don't need
train = train.drop(['Cabin', 'Name', 'PassengerId', 'groupId', 'personId'], axis=1)
test = test.drop(['Cabin', 'Name', 'PassengerId', 'groupId', 'personId'], axis=1)

In [ ]:
# Smart NA filling:
# CryoSleep passengers can't spend money — fill their spending with 0
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    train.loc[train['CryoSleep'] == True, col] = train.loc[train['CryoSleep'] == True, col].fillna(0)
    test.loc[test['CryoSleep'] == True, col] = test.loc[test['CryoSleep'] == True, col].fillna(0)

# Fill remaining NAs — median for numerical, mode for categorical (fit on train only)
fill_cols = [col for col in train.columns if col != 'Transported']
for col in fill_cols:
    if train[col].dtype in ['int64', 'float64']:
        median_val = train[col].median()
        train[col] = train[col].fillna(median_val)
        test[col] = test[col].fillna(median_val)
    else:
        mode_val = train[col].mode()[0]
        train[col] = train[col].fillna(mode_val)
        test[col] = test[col].fillna(mode_val)

print('NAs in train:', train.isna().sum().sum())
print('NAs in test:', test.isna().sum().sum())

In [ ]:
# Feature engineering — keep individual spending AND total
train['totalSpend'] = train[spend_cols].sum(axis=1)
test['totalSpend'] = test[spend_cols].sum(axis=1)

## 4. Encoding

In [ ]:
# Boolean and string-number columns to int
train[['RoomNum', 'CryoSleep', 'VIP']] = train[['RoomNum', 'CryoSleep', 'VIP']].astype(int)
test[['RoomNum', 'CryoSleep', 'VIP']] = test[['RoomNum', 'CryoSleep', 'VIP']].astype(int)

# One-hot encode categorical columns
train = pd.get_dummies(train, columns=['Side', 'Deck', 'Destination', 'HomePlanet'])
test = pd.get_dummies(test, columns=['Side', 'Deck', 'Destination', 'HomePlanet'])

# Fix whitespace in column names
train.columns = train.columns.str.replace(' ', '_')
test.columns = test.columns.str.replace(' ', '_')

# Align train and test columns
train, test = train.align(test, join='left', axis=1, fill_value=0)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Columns:', train.columns.tolist())

## 5. Modeling

In [ ]:
X = train.drop('Transported', axis=1)
y = train['Transported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Baseline: Logistic Regression
numerical_cols = ['Age', 'RoomNum', 'groupSize', 'totalSpend'] + spend_cols
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
print('Logistic Regression:')
print('  Train accuracy:', accuracy_score(y_train, lr.predict(X_train_scaled)))
print('  Test accuracy:', accuracy_score(y_test, lr.predict(X_test_scaled)))

In [ ]:
# LightGBM — best performing model
lgbm = lgb.LGBMClassifier(
    learning_rate=0.05,
    max_depth=8,
    n_estimators=100,
    num_leaves=31,
    random_state=42
)
lgbm.fit(X_train, y_train)
print('LightGBM:')
print('  Train accuracy:', accuracy_score(y_train, lgbm.predict(X_train)))
print('  Test accuracy:', accuracy_score(y_test, lgbm.predict(X_test)))

In [ ]:
# Cross-validation — more reliable performance estimate
cv_scores = cross_val_score(lgbm, X, y, cv=5, scoring='accuracy')
print('CV scores:', cv_scores)
print('Mean CV accuracy:', cv_scores.mean())

## 6. Submission

In [ ]:
# Train final model on all data
final_model = lgb.LGBMClassifier(
    learning_rate=0.05,
    max_depth=8,
    n_estimators=100,
    num_leaves=31,
    random_state=42
)
final_model.fit(X, y)

# Predict on test set
test_features = test.drop('Transported', axis=1)
final_predictions = final_model.predict(test_features)

# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': final_predictions
})
submission.to_csv('submission.csv', index=False)
print('Submission saved!')
print(submission.head())